# AI Customer Support Ticket Classifier

This notebook improves the prototype with a repeatable evaluation set, metrics, confidence-based review, failure-case analysis, and an optional Gradio interface.

**Model:** `facebook/bart-large-mnli` (zero-shot classification via Hugging Face Transformers)

**Categories:** Billing, Technical Issue, Account Access, General Inquiry

> Important: model scores are not calibrated probabilities. Treat them as relative confidence signals, not guaranteed correctness. The first run downloads model files and may require substantial RAM/compute and an internet connection.

In [ ]:
%pip install -q transformers torch scikit-learn pandas matplotlib gradio

## 1. Load the model and define labels

The candidate-label descriptions are kept consistent across examples so the evaluation is fair.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from transformers import pipeline

LABELS = ["Billing", "Technical Issue", "Account Access", "General Inquiry"]
MODEL_NAME = "facebook/bart-large-mnli"

classifier = pipeline(
    task="zero-shot-classification",
    model=MODEL_NAME
)

def classify_ticket(message, labels=LABELS):
    """Return predicted label, model score, and ranked label scores."""
    if not isinstance(message, str) or not message.strip():
        return {"label": "Needs more information", "score": 0.0, "scores": {}}
    result = classifier(message.strip(), candidate_labels=labels, multi_label=False)
    return {
        "label": result["labels"][0],
        "score": float(result["scores"][0]),
        "scores": {label: float(score) for label, score in zip(result["labels"], result["scores"])}
    }

print("Loaded:", MODEL_NAME)

## 2. Build a small labelled evaluation set

These are illustrative examples created for a demo, not a representative production dataset. Keep the test examples separate from any examples used to tune the model or labels.

In [ ]:
evaluation_data = [
    ("I was charged twice for my monthly subscription.", "Billing"),
    ("Please explain the extra fee on my invoice.", "Billing"),
    ("My refund has not arrived yet.", "Billing"),
    ("Can I change the card used for payments?", "Billing"),
    ("The app crashes when I upload a PDF.", "Technical Issue"),
    ("The dashboard shows a server error.", "Technical Issue"),
    ("The download button does nothing.", "Technical Issue"),
    ("The website freezes after I click Save.", "Technical Issue"),
    ("I forgot my password and cannot sign in.", "Account Access"),
    ("My account is locked after too many attempts.", "Account Access"),
    ("The password reset link has expired.", "Account Access"),
    ("I cannot access my profile.", "Account Access"),
    ("When is customer support open?", "General Inquiry"),
    ("Where can I read the product documentation?", "General Inquiry"),
    ("Which services does your company provide?", "General Inquiry"),
    ("Do you offer support on weekends?", "General Inquiry"),
]

eval_df = pd.DataFrame(evaluation_data, columns=["message", "actual"])
eval_df

## 3. Run evaluation and inspect metrics

In [ ]:
predictions = [classify_ticket(text) for text in eval_df["message"]]
eval_df["predicted"] = [p["label"] for p in predictions]
eval_df["score"] = [p["score"] for p in predictions]
eval_df["correct"] = eval_df["actual"] == eval_df["predicted"]

accuracy = accuracy_score(eval_df["actual"], eval_df["predicted"])
print(f"Demo-set accuracy: {accuracy:.1%} ({eval_df['correct'].sum()}/{len(eval_df)})")
print("\nClassification report:")
print(classification_report(
    eval_df["actual"], eval_df["predicted"], labels=LABELS,
    zero_division=0
))
display(eval_df)

In [ ]:
cm = confusion_matrix(eval_df["actual"], eval_df["predicted"], labels=LABELS)
display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=LABELS)
display.plot(xticks_rotation=35, values_format="d")
plt.title("Confusion matrix — demo evaluation set")
plt.tight_layout()
plt.show()

## 4. Failure cases and review policy

A model can confidently choose the wrong label, especially when a message mentions multiple issues, is vague, or falls outside the four categories. These examples are **tests to run**, not claims that the model will fail in a specific way.

For this prototype, route low-scoring or out-of-scope messages to a human rather than automatically sending them to a team. The threshold below is a demonstration setting and must be tuned on a larger, labelled validation set.

In [ ]:
failure_tests = [
    ("I was charged twice and now I cannot log in.", "Ambiguous / multiple intents"),
    ("It does not work.", "Vague message"),
    ("Please cancel my subscription and refund the last payment.", "Multiple intents"),
    ("I want to speak to a human about a complaint.", "Potentially out of scope"),
    ("My invoice page crashes when I open it.", "Billing vs technical overlap"),
]

REVIEW_THRESHOLD = 0.65  # illustrative only; tune using validation data
failure_rows = []
for message, risk in failure_tests:
    result = classify_ticket(message)
    action = "Human review" if result["score"] < REVIEW_THRESHOLD else "Suggested category; verify"
    failure_rows.append({
        "message": message,
        "risk to inspect": risk,
        "predicted": result["label"],
        "model score (not calibrated)": round(result["score"], 3),
        "handling": action,
    })
pd.DataFrame(failure_rows)

## 5. Prompt/model comparison experiment

Zero-shot classification depends on label wording. Compare the original labels with more descriptive candidate labels and evaluate both on the **same** examples. Do not pick the wording solely because it performs best on this tiny test set; confirm on a larger held-out set.

In [ ]:
DESCRIPTIVE_LABELS = [
    "Billing and payments, including invoices, charges, and refunds",
    "Technical problems, bugs, errors, crashes, or broken features",
    "Account access, login, password reset, or locked account",
    "General questions about services, documentation, or business hours",
]
LABEL_MAP = dict(zip(DESCRIPTIVE_LABELS, LABELS))

def compare_label_wording(messages):
    rows = []
    for message, actual in messages:
        plain = classify_ticket(message, LABELS)
        descriptive = classify_ticket(message, DESCRIPTIVE_LABELS)
        descriptive_label = LABEL_MAP[descriptive["label"]]
        rows.append({
            "message": message,
            "actual": actual,
            "plain prediction": plain["label"],
            "plain correct": plain["label"] == actual,
            "descriptive prediction": descriptive_label,
            "descriptive correct": descriptive_label == actual,
        })
    comparison = pd.DataFrame(rows)
    summary = pd.DataFrame([
        {"candidate-label wording": "Short labels", "accuracy": comparison["plain correct"].mean()},
        {"candidate-label wording": "Descriptive labels", "accuracy": comparison["descriptive correct"].mean()},
    ])
    return comparison, summary

comparison_df, comparison_summary = compare_label_wording(evaluation_data)
display(comparison_summary)
display(comparison_df)

## 6. Simple interactive Gradio interface

Run this cell to create a small local web app. If running in a hosted notebook, Gradio may provide a temporary share URL when `share=True`; avoid sharing sensitive customer messages.

In [ ]:
import gradio as gr

def predict_for_ui(message):
    if not message or not message.strip():
        return {"status": "Enter a message", "category": "—", "score": "—", "all scores": {}}
    result = classify_ticket(message)
    status = "Human review recommended" if result["score"] < REVIEW_THRESHOLD else "Suggested category — verify before routing"
    return {
        "status": status,
        "category": result["label"],
        "score (not calibrated)": round(result["score"], 3),
        "all label scores": result["scores"],
    }

demo = gr.Interface(
    fn=predict_for_ui,
    inputs=gr.Textbox(lines=3, label="Customer support message", placeholder="Example: I was charged twice this month."),
    outputs=gr.JSON(label="AI suggestion"),
    title="Customer Support Ticket Classifier",
    description="Demo prototype. Predictions can be wrong; review before routing. Do not enter personal or sensitive information.",
    examples=[
        ["I was charged twice this month."],
        ["The app crashes when I upload a file."],
        ["I forgot my password and cannot sign in."],
        ["What are your customer support hours?"],
        ["It does not work."],
    ],
)

# In a local environment use demo.launch(). In a hosted notebook, demo.launch(share=True)
# may generate a temporary public URL. Do not use public sharing with sensitive data.
demo.launch()

## 7. Success criteria and limitations

Suggested initial targets: accuracy ≥ 85%, macro F1 ≥ 0.80, and recall ≥ 0.75 for every category. Report the measured values rather than assuming they are achieved. This tiny demonstration set is too small to establish reliable real-world performance.

**Limitations:**
- The model's scores are not calibrated probabilities.
- The test set is small and hand-written, so it may not reflect real customer language.
- The four labels do not cover every possible request.
- Multiple-intent messages are forced into one category.
- Human review is recommended for low-confidence, ambiguous, or out-of-scope messages.

**Next steps:** collect consented/de-identified real examples, create a stratified held-out test set, label examples with clear guidelines, inspect errors by category, tune the review threshold on validation data, and re-evaluate before deployment.